In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from scipy.stats import multivariate_normal
from scipy.spatial.transform import Rotation
from scipy.linalg import block_diag
import pinocchio as pin
import scipy.special as sp
import scipy.linalg as la
import time
import plotly.graph_objects as go

In [ ]:
                             # ====================================================================
                             #               Lie Group Extended Kalman Filter                     #
                             #                       Basic Functions                              #
                             # ====================================================================

# =============================================================
#  1) Synthetic Data Generation
# =============================================================

# This function generates the synthetic data that is needed for the algorithm

def Synthetic_Data_Generation(dt,T_total,xi1,xi2,slope,sigma_a,sigma_w,sigma_ba,sigma_bw,sigma_eps_s1,
                             sigma_eps_s2,sigma_eps_s3,sigma_eps_s4,L1,L2,L3,L4,pos_start,vel_start):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    dt--------------> sampling time
    T_total---------> Total time of simulation
    xi1-------------> Autoregressive Parameter of the acceleration bias
    xi2-------------> Autoregressive Parameter of the angular velocity bias
    slope-----------> Slope angle of the aircraft (standard aviation = 3 degrees)
    sigma_a---------> Standard deviation of the acceleration
    sigma_w---------> Standard deviation of the angular velocity
    sigma_ba--------> Standard deviation of the bias accelerometer
    sigma_bw--------> Standard deviation of the bias gyrometer
    sigma_eps_s1----> Standard deviation of the measurement associated with Landmark 1
    sigma_eps_s2----> Standard deviation of the measurement associated with Landmark 2
    sigma_eps_s3----> Standard deviation of the measurement associated with Landmark 3
    sigma_eps_s4----> Standard deviation of the measurement associated with Landmark 4
    L1--------------> Position of the Landmark 1 (inertial frame)
    L2--------------> Position of the Landmark 2 (inertial frame)
    L3--------------> Position of the Landmark 3 (inertial frame)
    L4--------------> Position of the Landmark 4 (inertial frame)
    pos_start-------> Initial position of the aircraft (inertial frame)
    vel_start-------> Initial velocity of the aircraft (inertial frame) 

    ===================
    OUTPUT PARAMETERS
    ===================
    P_true----------> Trajectory of the aircraft during the 115 seconds (Is a matrix 3 X N)
    V_true----------> Velocity of the aircraft during the 115 seconds (Is a matrix 3 X N)
    M_rotations-----> Sequence of rotation matrices during the 115 seconds (Is a tensor N X 3 X 3)
    bias_a----------> Sequence of bias of acceleration sensor during the 115 seconds (Is a matrix 3 X N)
    bias_w----------> Sequence of bias of gyroscopes during the 115 seconds (Is a matrix 3 X N)
    acc_m-----------> Measurements of the accelerometer (Is a matrix 3 X N)
    gyro_m----------> Measurements of the gyroscope (Is a matrix 3 X N)
    y_s1------------> Camera Measurements associated with Landmark 1 (Is a matrix 3 X N)
    y_s2------------> Camera Measurements associated with Landmark 2 (Is a matrix 3 X N)
    y_s3------------> Camera Measurements associated with Landmark 3 (Is a matrix 3 X N)
    y_s4------------> Camera Measurements associated with Landmark 4 (Is a matrix 3 X N)
    ILS_LOC---------> Localizer Measurements (Is a matrix 3 X N)
    ILS_GS----------> Glide Slope Measurements (Is a matrix 3 X N)   
    '''

    # Constants adopted for the model
    g_n = np.array([0, 0, 9.80665]) 
    
    # Number of samples to be used
    N = int(T_total / dt)             

    # Time axis
    time = np.linspace(0, T_total, N) 

    # ============================================
    # Structures to store the data - Hidden States
    # ============================================
    # Our hidden state is {pn,vn,Rn,abn,wbn} -> 4 vectors in R3 and matrix in SO(3)
    P_true = np.zeros((3, N))
    V_true = np.zeros((3, N))
    M_rotations = np.zeros((N,3,3))
    bias_a = np.zeros((3, N))
    bias_w = np.zeros((3, N))
    
    # ============================================
    # Structures to store the data - Measurements
    # ============================================
    # In real life we would have only the measurements acquired by the IMU sensors and the cameras
    # 1. IMU sensors: accelerometer and gyroscope
    acc_m = np.zeros((3, N))
    gyro_m = np.zeros((3, N))

    # 2. Camera measurements - 4 landmarks in the soil
    y_s1 = np.zeros((3, N))
    y_s2 = np.zeros((3, N))
    y_s3 = np.zeros((3, N))
    y_s4 = np.zeros((3, N))

    # ============================================
    # Structures to store the results - LOC / GS
    # ============================================
    ILS_LOC = np.zeros(N)
    ILS_GS = np.zeros(N)
     
    # ==================================================
    # Filling the data structures - Hidden states (n = 0)
    # ==================================================

    # 1) Hidden state - position
    # The first point is known - last point with GNSS signal
    P_true[:,0] = pos_start

    # 2) Hidden state - velocity
    # Under the hypothesis of constant velocity
    for k in range(N):
        V_true[:,k] = vel_start

    # 3) Hidden state - Rotation Matrix
    # Rotation Matrix
    R_0 = Rot_Euler_angles(vel_start)
    M_rotations[0,:,:] = R_0

    # 4 & 5) Hidden state - Biases
    bias_a[:,0] = np.random.normal(0, sigma_ba, 3).flatten()
    bias_w[:,0] = np.random.normal(0, sigma_bw, 3).flatten()

    # ==================================================
    # Filling the data structures - Measurements (n = 0)
    # ==================================================

    # IMU emulation
    acc_m[:, 0] = M_rotations[0,:,:].T @ (-g_n) + bias_a[:, 0] + np.random.normal(0, sigma_a, 3)
    gyro_m[:, 0] = 0.0 + bias_w[:, 0] + np.random.normal(0, sigma_w, 3)
        
    # Camera measurements in the first time instant (n = 0)
    y_s1[:, 0] = (R_0.T @ Model_Camera(L1 - P_true[:,0].reshape(3,1)) + \
                  np.random.normal(0, sigma_eps_s1, 3).reshape(3,1)).flatten()
    
    y_s2[:, 0] = (R_0.T @ Model_Camera(L2 - P_true[:,0].reshape(3,1)) + \
                  np.random.normal(0, sigma_eps_s2, 3).reshape(3,1)).flatten()
    
    y_s3[:, 0] = (R_0.T @ Model_Camera(L3 - P_true[:,0].reshape(3,1)) + \
                  np.random.normal(0, sigma_eps_s3, 3).reshape(3,1)).flatten()
    
    y_s4[:, 0] = (R_0.T @ Model_Camera(L4 - P_true[:,0].reshape(3,1)) + \
                  np.random.normal(0, sigma_eps_s4, 3).reshape(3,1)).flatten()
    
    # ==================================================
    # Filling the data structures - Resutls (n = 0)
    # ==================================================

    ILS_n0 = ILS_Calculator(pos_start[0],pos_start[1],pos_start[2],slope)
    ILS_LOC[0] = ILS_n0[0]
    ILS_GS[0] = ILS_n0[1]

    # ==================================================
    # Simulation time for k >= 1
    # ==================================================
    for k in range(1,N):
        # Position Integration 
        P_true[:, k] = P_true[:, k-1] + V_true[:, k-1] * dt
        
        # --- Emulate IMU Measurements ---
        # From State Eq: v_{n+1} = v_n + [R_n (a_m - a_b - eps_a) + g] * dt [cite: 67]
        # Since V is constant, v_{n+1} - v_n = 0. 
        # Therefore: R_n (a_m - a_b - eps_a) = -g  =>  a_m = R_n^T(-g) + a_b + eps_a

        # Since attitude is constant, true angular rate is 0 [cite: 69-71]
        # Omega_m = Omega_true + bias_w + eps_w

        # Bias generation - Autoregressive Process AR(1)
        bias_a[:,k] = xi1*bias_a[:,k - 1] + np.random.normal(0, sigma_ba, 3)
        bias_w[:,k] = xi2*bias_w[:,k - 1] + np.random.normal(0, sigma_bw, 3)

        # Angular Velocity coming from the IMU
        gyro_m[:, k] = 0.0 + bias_w[:, k] + np.random.normal(0, sigma_w, 3)
        
        # Rotation Matrix is constant (angular velocity is constant)
        M_rotations[k,:,:] = M_rotations[k - 1,:,:]

        # Acceleration coming from the IMU
        acc_m[:, k] = M_rotations[k,:,:].T @ (-g_n) + bias_a[:, k] + np.random.normal(0, sigma_a, 3)
        
        # ILS calculation
        ILS_mesurement = ILS_Calculator(P_true[0,k],P_true[1,k],P_true[2,k],slope)
        ILS_LOC[k] = ILS_mesurement[0]
        ILS_GS[k] = ILS_mesurement[1]

        # Camera acquisitions
        y_s1[:, k] = (M_rotations[k,:,:].T @ Model_Camera(L1 - P_true[:,k].reshape(3,1)) + \
                      np.random.normal(0, sigma_eps_s1, 3).reshape(3,1)).flatten()

        y_s2[:, k] = (M_rotations[k,:,:].T @ Model_Camera(L2 - P_true[:,k].reshape(3,1)) +\
                      np.random.normal(0, sigma_eps_s2, 3).reshape(3,1)).flatten()

        y_s3[:, k] = (M_rotations[k,:,:].T @ Model_Camera(L3 - P_true[:,k].reshape(3,1)) + \
                      np.random.normal(0, sigma_eps_s3, 3).reshape(3,1)).flatten()

        y_s4[:, k] = (M_rotations[k,:,:].T @ Model_Camera(L4 - P_true[:,k].reshape(3,1)) + \
                      np.random.normal(0, sigma_eps_s4, 3).reshape(3,1)).flatten()

    print("Simulation of data finished")
    print("Running the Filter...")
    return P_true, V_true, M_rotations, bias_a, bias_w, acc_m, gyro_m, y_s1, y_s2, y_s3, y_s4, ILS_LOC, ILS_GS

# =============================================================
# 2) Camera Model
# =============================================================

# Depending on the model of the camera, we will adopt certain non-linearities
def Model_Camera(x):
    # In this very simple model, there is no non-linearity
    return x

# =============================================================
# 3) Plot Function
# =============================================================

def Plot_3D_trajectory(P):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    P ---> Real trajectory of the aircraft

    ===================
    OUTPUT PARAMETERS
    ===================
    Plots
    '''
    # ===========================================================================
    # Interactive Landing Trajectory - Ground Landmarks
    # ===========================================================================
    
    fig = go.Figure()
    
    # Aircraft Trajectory Line
    fig.add_trace(go.Scatter3d(
        x=P[0, :],
        y=P[1, :],
        z=P[2, :],
        mode='lines',
        name='Landing Trajectory',
        line=dict(width=4, color='blue')
    ))
    
    # Coordinate Origin (Red Marker)
    fig.add_trace(go.Scatter3d(
        x=[0],
        y=[0],
        z=[0],
        mode='markers',
        name='Origin', 
        marker=dict(size=6, color='red')
    ))
    
    # Landmark 1
    fig.add_trace(go.Scatter3d(
        x=[L1[0,0]],
        y=[L1[1,0]],
        z=[L1[2,0]],
        mode='markers',
        name='L1',
        marker=dict(size=6, color='green')
    ))
    
    # Landmark 2
    fig.add_trace(go.Scatter3d(
        x=[L2[0,0]],
        y=[L2[1,0]],
        z=[L2[2,0]],
        mode='markers',
        name='L2',
        marker=dict(size=6, color='yellow')
    ))
    
    # Landmark 3
    fig.add_trace(go.Scatter3d(
        x=[L3[0,0]],
        y=[L3[1,0]],
        z=[L3[2,0]],
        mode='markers',
        name='L3',
        marker=dict(size=6, color='black')
    ))
    
    # Landmark 4
    fig.add_trace(go.Scatter3d(
        x=[L4[0,0]],
        y=[L4[1,0]],
        z=[L4[2,0]],
        mode='markers',
        name='L4',
        marker=dict(size=6, color='purple')
    ))
    
    fig.update_layout(
        width=800,     # Figure width in pixels
        height=800,    # Figure height in pixels
        title="Interactive 3D Trajectory - NED Coordinates",
        scene=dict(
            xaxis_title='North [m]',
            yaxis_title='East [m]',
            zaxis_title='Down [m]',
            zaxis=dict(autorange='reversed'),
            xaxis=dict(autorange='reversed')
        )
    )
    
    
    # Plot
    fig.show()
    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(P[0, :], P[1, :], P[2, :], color='blue', linewidth=2, label='Landing Trajectory')
    ax.scatter(0, 0, 0, color='red', s=40, label='Origin')
    ax.scatter(L1[0,0], L1[1,0], L1[2,0], color='green', s=40, label='L1')
    ax.scatter(L2[0,0], L2[1,0], L2[2,0], color='yellow', edgecolor='black', s=40, label='L2')
    ax.scatter(L3[0,0], L3[1,0], L3[2,0], color='black', s=40, label='L3')
    ax.scatter(L4[0,0], L4[1,0], L4[2,0], color='purple', s=40, label='L4')
    ax.set_title("3D Trajectory - NED Coordinates")
    ax.set_xlabel('North (x) [m]')
    ax.set_ylabel('East (y) [m]')
    ax.set_zlabel('Down (z) [m]')
    ax.invert_xaxis()
    ax.invert_zaxis()
    ax.legend()
    
    # ===========================================================================
    # Save image, if needed
    # ===========================================================================
    plt.savefig('trajectory_matplotlib.pdf', format='pdf')
    
    # Mostra o gráfico na tela
    plt.show()
    
    print("=========================================")
    print("PDF generated")
    print("=========================================")

                             # ====================================================================
                             #               Lie Group Extended Kalman Filter                     #
                             #                 Functions for the processing                       #
                             # ====================================================================


# ====================================================================
# 4) Model Matrices Construction - Matrix F
# ====================================================================

def Fn_Calculator(Rnm1_nm1,am_nm1,abnm1_nm1,omega_nm1,xi1,xi2):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    Rnm1_nm1----> Previous Rotation Matrix
    am_nm1------> Previous acceleration measurement
    abnm1_nm1---> Previous acceleration bias measurement (estimated)
    omega_nm1---> Previous angular velocity measurement
    xi1---------> AR(1) parameter (acceleration bias)
    xi2---------> AR(1) parameter (angular velocity bias)
    
    ===================
    OUTPUT PARAMETERS
    ===================
    Fn - Matrix 15 X 15, the State Transition Matrix
    '''
    # State transition matrix is 15 x 15
    F = np.zeros((15,15))

    # First Row
    F[0:3,0:3] = np.eye(3)
    F[0:3,3:6] = dt*np.eye(3)
    F[0:3,6:9] = -0.5*dt**2 * Rnm1_nm1 @ CartesianSpace2LieAlgebra_Wedge(am_nm1 - abnm1_nm1)
    F[0:3,9:12] = -0.5*Rnm1_nm1*dt**2

    # Second Row
    F[3:6,3:6] = np.eye(3)
    F[3:6,6:9] = -dt * Rnm1_nm1 @ CartesianSpace2LieAlgebra_Wedge(am_nm1 - abnm1_nm1)
    F[3:6,9:12] = -Rnm1_nm1*dt
    
    # Third Row
    F[6:9,6:9] = exp_map_from_so3_to_SO3(CartesianSpace2LieAlgebra_Wedge(omega_nm1*dt)).T
    F[6:9,12:15] = -dt*Left_Jacobian_Calculator(omega_nm1*dt)

    # Fourth Row
    F[9:12,9:12] = xi1 * np.eye(3)

    # Fifth Row
    F[12:15,12:15] = xi2 * np.eye(3)

    return F

# ====================================================================
# 5) Model Matrices Construction - Matrix L_G
# ====================================================================

# Left Jacobian included in the structure of the Group
def LG_Calculator(dt, wn_hat):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    dt------> Sampling time
    wn_hat--> Difference between the raw measurement of angular velocity and the bias
    
    ===================
    OUTPUT PARAMETERS
    ===================
    LG - Matrix 15 X 15, the Left Jacobian on the Lie group
    '''
    LG = np.zeros((15,15))
    I3 = np.eye(3)
    leftJ = Left_Jacobian_Calculator(wn_hat*dt)
    LG = block_diag(I3, I3, leftJ, I3, I3)
    return LG
    
# ====================================================================
# 6) Model Matrices Construction - Matrix Qn
# ====================================================================

def Qn_Calculator(Rnm1_nm1,S_a,S_omega,S_ba,S_bw):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    Rnm1_nm1----> Previous Rotation Matrix
    S_a---------> Covariance Matrix of the noise acceleration
    S_omega-----> Covariance Matrix of the noise angular velocity
    S_ba--------> Covariance Matrix of the noise bias acceleration
    S_bw--------> Covariance Matrix of the noise bias angular velocity
    
    ===================
    OUTPUT PARAMETERS
    ===================
    Qn - Matrix 15 X 15, the Covariance Matrix of the complete hidden state
    '''
    Qn = np.zeros((15,15))

    # First Row of the Matrix
    Qn[0:3,0:3] = 0.25*(dt**4)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T
    Qn[0:3,3:6] = 0.5*(dt**3)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T

    # Second Row of the Matrix
    Qn[3:6,0:3] = 0.5*(dt**3)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T
    Qn[3:6,3:6] = (dt**2)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T

    # Third Row of the Matrix
    Qn[6:9,6:9] = (dt**2)*S_omega

    # Fourth Row of the Matrix
    Qn[9:12,9:12] = S_ba

    # Fifth Row of the Matrix
    Qn[12:15,12:15] = S_bw

    return Qn

# ====================================================================
# 7) Model Matrices Construction - Matrix H
# ====================================================================

def Hn_Calculator(Rn_nm1,L1,L2,L3,L4,pn_nm1):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    Rnm1_nm1----> Previous Rotation Matrix
    L1----------> Position of the Landmark 1 (Inertial Frame)
    L2----------> Position of the Landmark 2 (Inertial Frame)
    L3----------> Position of the Landmark 3 (Inertial Frame)
    L4----------> Position of the Landmark 4 (Inertial Frame)
    pn_nm1------> Predicted Position (output of Kalman Predictor)
    
    ===================
    OUTPUT PARAMETERS
    ===================
    Hn - Matrix 12 X 15, Jacobian matrix of the observation model
    '''
    Hn = np.zeros((12,15))

    # First block of 3 rows
    Hn[0:3,0:3] = -Rn_nm1.T
    Hn[0:3,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L1 - pn_nm1))

    # Second block of 3 rows
    Hn[3:6,0:3] = -Rn_nm1.T
    Hn[3:6,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L2 - pn_nm1))

    # Third block of 3 rows
    Hn[6:9,0:3] = -Rn_nm1.T
    Hn[6:9,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L3 - pn_nm1))

    # Fourth block of 3 rows
    Hn[9:12,0:3] = -Rn_nm1.T
    Hn[9:12,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L4 - pn_nm1))

    return Hn

# ====================================================================
# 8) Construction of the Matrix that accumulates innovations from the 4 Landmarks
# ====================================================================

def Inovation_Calculator(y1, y2, y3, y4, Rn_nm1, L1, L2, L3, L4, pn_nm1):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    Rnm1_nm1----> Previous Rotation Matrix
    y1----------> Camera measurement associated with Landmark 1
    y2----------> Camera measurement associated with Landmark 2
    y3----------> Camera measurement associated with Landmark 3
    y4----------> Camera measurement associated with Landmark 4
    L1----------> Position of the Landmark 1 (Inertial Frame)
    L2----------> Position of the Landmark 2 (Inertial Frame)
    L3----------> Position of the Landmark 3 (Inertial Frame)
    L4----------> Position of the Landmark 4 (Inertial Frame)
    pn_nm1------> Predicted Position (output of Kalman Predictor)
    
    ===================
    OUTPUT PARAMETERS
    ===================
    inov - Matrix 12 X 1, Stacked Inovation Matrix
    '''
    inov = np.zeros((12,1))
    inov[0:3,:]  = y1 - Rn_nm1.T @ (L1 - pn_nm1) 
    inov[3:6,:]  = y2 - Rn_nm1.T @ (L2 - pn_nm1) 
    inov[6:9,:]  = y3 - Rn_nm1.T @ (L3 - pn_nm1) 
    inov[9:12,:] = y4 - Rn_nm1.T @ (L4 - pn_nm1) 
    return inov

# ====================================================================
# 9) Function to split the vector in R15 into 5 blocks in R3
# ====================================================================

def Split_Correction_Vector(vector_R15):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    vector_R15 ---> Vector with 15 numbers

    ===================
    OUTPUT PARAMETERS
    ===================
    5 vectors each one in R3
    '''
    
    a = vector_R15[0:3]
    b = vector_R15[3:6]
    c = vector_R15[6:9]
    d = vector_R15[9:12]
    e = vector_R15[12:15]
    return a,b,c,d,e

# ====================================================================
# 10) Function to find the Kalman gain
# ====================================================================

def Kalman_Gain_Calculator(Pn_nm1,Hn,Rn):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    Pn_nm1----------> Predicted Covariance Matrix (output of Kalman Predictor)
    Hn--------------> Jacobian Matrix of the Observation Model
    Rn--------------> Covariance Matrix of the Observation Noise

    ===================
    OUTPUT PARAMETERS
    ===================
    Kalman Gain
    '''
    Sxx = Pn_nm1 @ Hn.T
    Syy = Hn @ Pn_nm1 @ Hn.T + Rn
    K = Sxx @ np.linalg.inv(Syy)
    return K

# ====================================================================
# H) Functions Regarding Lie Algebra/Group
# ====================================================================

# ====================================
# 11) Wedge operator: R3 -> so(3) 
# ====================================
def CartesianSpace2LieAlgebra_Wedge(vector):
    # We receive the vector in R3
    x,y,z = vector.flatten()
    return np.array([[ 0,  -z,   y],
                     [ z,   0,  -x],
                     [-y,   x,   0]])

# ====================================
# 12) Vee operator: so(3) -> R3 
# ====================================
def LieAlgebra2CartesianSpace_Vee(X):
    # We receive a skew symmetric Matrix in the Lie Algebra
    return np.array([-X[1,2],X[0,2],-X[0,1]]).reshape(3,1)

# ====================================
# 13) exp map: so(3) -> SO(3)
# ====================================
def exp_map_from_so3_to_SO3(X):
    theta = np.sqrt(0.5 * np.trace(X.T @ X))

    if theta < 1e-4:
        # Taylor Series Approximation
        return np.eye(3) + X + (X @ X)/2
    else:
        # Rodrigues Formula for computing the matrix exponential of a skew-symmetric Matrix
        return np.eye(3) + (np.sin(theta) / theta) * X + ((1 - np.cos(theta)) / theta**2) * (X @ X)

# ====================================
# 14) log map: SO(3) -> so(3)
# ====================================
def log_map_from_SO3_to_so3(X):
    # Force the outputs with clip function, between -1 and 1
    cos_theta = np.clip((np.trace(X) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    
    if theta < 1e-4:
        # Taylor Series Approximation again
        return (X - X.T)/2
    else:
        return (0.5 * theta / np.sin(theta)) * (X - X.T)

# ====================================
# 15) Left Jacobian Phi_SO3
# ====================================
def Left_Jacobian_Calculator(v_theta):
    # Note that v_theta belongs to R3
    norm = np.linalg.norm(v_theta)
    v_LieAlg = CartesianSpace2LieAlgebra_Wedge(v_theta)  
    
    if norm < 10**(-4):
        return np.eye(3) + 0.5*v_LieAlg + (1/6)*(v_LieAlg @ v_LieAlg)
    else:
        return np.eye(3) + ((1 - np.cos(norm))/norm**2)*v_LieAlg +\
        ((norm - np.sin(norm))/norm**3)*(v_LieAlg @ v_LieAlg)
        
# ====================================
# 16) Adjoint Operator in SO(3)
# ====================================
def Adj_SO3(vector):
    I = np.eye(3)
    left = Left_Jacobian_Calculator(vector)
    return block_diag(I,I,left,I,I)
    
# ====================================================================
# 17) ILS Generation - LOC and GS
# ====================================================================

# Monte Carlo Approach
def Generator_ILS_Gaussian_Mixture(v_mean, M_cov, sample_size, glide_slope_incline):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    v_mean-----------------> A posteriori estimated mean of the 5 states (is a vector 15 X 1)
    M_cov------------------> A posteriori estimated covariance matrix of the 5 states (is a matrix 15 X 15)
    sample_size------------> Number of Monte Carlo Experiments
    glide_slope_incline----> 3 degrees, according to the international rules
    
    ===================
    OUTPUT PARAMETERS
    ===================
    2 real numbers, the estimated LOC (Localizer) and estimated GS (Glide Slope)
    '''
    
    ILS_LOC = 0
    ILS_GS = 0
    
    for i in range(sample_size):
        z_sampled = np.random.multivariate_normal(v_mean.flatten(), M_cov)
                                                
        # Call the function to calculate ILS
        ILS_estimated = ILS_Calculator(z_sampled[0], z_sampled[1], z_sampled[2], glide_slope_incline)
        ILS_LOC = ILS_LOC + ILS_estimated[0]
        ILS_GS = ILS_GS + ILS_estimated[1]

    # Must divide by sample_size to obtain the mean value
    return [ILS_LOC/sample_size, ILS_GS/sample_size]
    

# ====================================================================
# 18) ILS Calculator
# ====================================================================

def ILS_Calculator(x, y, z, glide_slope_incline):
    # Localizer
    LOC = y

    # Glide slope
    GS = np.rad2deg(np.atan(np.abs(z)/np.sqrt(x**2 + y**2))) - glide_slope_incline
    return [LOC, GS]

# ====================================================================
# 19) Rotation Matrix from Euler Angles
# ====================================================================

def Rot_Euler_angles(vel_start):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    vel_start ----> Is the vector in R3, containing the velocity of the aircraft
    
    ===================
    OUTPUT PARAMETERS
    ===================
    Rotation matrix obtained by euler angles (Is a 3 X 3 Matrix)
    '''
    # Determine constant attitude (Pitch and Yaw) from velocity vector
    vx, vy, vz = vel_start
    yaw = np.arctan2(vy, vx)
    pitch = np.arctan2(-vz, np.sqrt(vx**2 + vy**2)) # -vz because Down is positive in NED
    roll = 0.0 

    # R_n is the rotation matrix from Body to Navigation (World) frame
    # Scipy Rotation uses scalar-last quaternions by default.
    rot = Rotation.from_euler('ZYX', [yaw, pitch, roll], degrees=False)
    return rot.as_matrix() 

# ====================================================================
# 20) Lie Group Extended Kalman Filter
# ====================================================================

def Lie_Group_Extended_Kalman_Filter(dt,T_total,y_s1,y_s2,y_s3,y_s4,acc_m,gyro_m,pos_start,
                                     vel_start,P0,Rn,L1,L2,L3,L4,sample_size,slope,Real_ILS_LOC,Real_ILS_GS):
    '''
    ===================
    INPUT PARAMETERS
    ===================
    dt---------------> Sampling Time
    T_total----------> Landing Time
    y_s1-------------> Entire sequence of camera measurements associated with Landmark 1 
    y_s2-------------> Entire sequence of camera measurements associated with Landmark 2 
    y_s3-------------> Entire sequence of camera measurements associated with Landmark 3 
    y_s4-------------> Entire sequence of camera measurements associated with Landmark 4
    acc_m------------> Entire sequence of acceleration measurements
    gyro_m-----------> Entire sequence of angular velocity measurements
    pos_start--------> Initial Position of the aircraft
    vel_start--------> Initial Velocity of the aircraft
    P0---------------> Initial Covariance Matrix
    Rn---------------> Covariance Matrix of the observation noises
    L1---------------> Position of the Landmark 1 in the inertial frame
    L2---------------> Position of the Landmark 2 in the inertial frame
    L3---------------> Position of the Landmark 3 in the inertial frame
    L4---------------> Position of the Landmark 4 in the inertial frame
    sample_size------> Number of samples to calculate the ILS values via Monte Carlo approach
    slope------------> Glide Slope of the aircraft
    Real_ILS_LOC-----> Real value of the localizer
    Real_ILS_GS------> Real value of the Glide Slope
    
    ===================
    OUTPUT PARAMETERS
    ===================
    Estimated sequence of
    1) Positions
    2) Velocities
    3) Rotation Matrices
    4) Acceleration Biases
    5) Angular Velocity Biases
    6) ILS LOC square error
    7) ILS GS square error
    8) A posteriori covariance matrices
    '''
    # Number of samples to be used
    N = int(T_total / dt)  

    # Initial Rotation Matrix
    R_start = Rot_Euler_angles(vel_start)
    
    # Constants adopted for the model
    g = np.array([0, 0, 9.80665]).reshape(3,1) 
    
    # =====================================================
    # Storage of the Information
    # =====================================================
    
    # 5 Hidden states to be estimated 
    M_pn_n_est = np.zeros((N,3,1))                         # Sequence of pn|n
    M_vn_n_est = np.zeros((N,3,1))                         # Sequence of vn|n
    M_Rn_n_est = np.zeros((N,3,3))                         # Sequence of Rn|n
    M_bias_a_n_n = np.zeros((N,3,1))                       # Sequence of abn|n
    M_bias_w_n_n = np.zeros((N,3,1))                       # Sequence of wbn|n
    
    # Full-state Covariance Matrix
    M_Pnn = np.zeros((N,15,15))     
    M_Pnn[0,:,:] = P0
    
    # LOC and GS estimation
    M_ILS_LOC_err = np.zeros(N)              
    M_ILS_GS_err = np.zeros(N) 
    
    # Initial State is
    M_pn_n_est[0,:,:]   = pos_start.reshape(3,1)
    M_vn_n_est[0,:,:]   = vel_start.reshape(3,1)
    M_Rn_n_est[0,:,:]   = R_start
    M_bias_a_n_n[0,:,:] = np.random.normal(0,sigma_ba, 3).reshape(3,1)
    M_bias_w_n_n[0,:,:] = np.random.normal(0,sigma_bw, 3).reshape(3,1)

    for i in range(1, N):

        # =========================================
        # Measurements available for each loop
        # =========================================
        # Measurements available at time i, note that yn_si has a 3x1 dimension
        yn_s1 = y_s1[:,i].reshape(3,1)
        yn_s2 = y_s2[:,i].reshape(3,1)
        yn_s3 = y_s3[:,i].reshape(3,1)
        yn_s4 = y_s4[:,i].reshape(3,1)
            
        # Accelerometer and gyroscope measurements
        amnm1 = acc_m[:,i - 1].reshape(3,1)
        wmnm1 = gyro_m[:,i - 1].reshape(3,1)
    
        # =========================================================
        # 1) EKF PREDICTION STEP
        # =========================================================
        # Divided into Mean and Covariance
    
        # ===================
        # Mean Prediction
        # ===================
        M_pn_n_est[i,:,:] = M_pn_n_est[i - 1,:,:] + M_vn_n_est[i - 1,:,:]*dt + \
                            (0.5*dt**2)*(g + M_Rn_n_est[i - 1] @ (amnm1 - M_bias_a_n_n[i - 1,:,:]))
    
        M_vn_n_est[i,:,:] = M_vn_n_est[i - 1,:,:] + \
                           dt*(g + M_Rn_n_est[i - 1] @ (amnm1 - M_bias_a_n_n[i - 1,:,:]))
    
        M_Rn_n_est[i,:,:] = M_Rn_n_est[i - 1,:,:] @ \
                    exp_map_from_so3_to_SO3(CartesianSpace2LieAlgebra_Wedge(dt*(wmnm1 - M_bias_w_n_n[i - 1,:,:])))
    
        M_bias_a_n_n[i,:,:] = xi1*M_bias_a_n_n[i - 1,:,:]
    
        M_bias_w_n_n[i,:,:] = xi2*M_bias_w_n_n[i - 1,:,:]
    
        # Calculation of Matrices F, Lg, and Qm1
        F = Fn_Calculator(M_Rn_n_est[i - 1,:,:],amnm1,M_bias_a_n_n[i - 1,:,:],wmnm1 - M_bias_w_n_n[i - 1,:,:],xi1,xi2)
        LG = LG_Calculator(dt, wmnm1 - M_bias_w_n_n[i - 1,:,:])
        Qnm1 = Qn_Calculator(M_Rn_n_est[i - 1,:,:],Sa,Sw,Sba,Sbw)
    
        # ===================
        # Covariance Matrix Prediction
        # ===================
        M_Pnn[i,:,:] = F @ M_Pnn[i - 1,:,:] @ F.T + LG @ Qnm1 @ LG.T
    
        # =========================================================
        # 2) EKF UPDATE STEP
        # =========================================================
       
        # ===================
        # Mean Update
        # ===================
        # 1) Innovation calculation
        mu_n = Inovation_Calculator(yn_s1, yn_s2, yn_s3, yn_s4, M_Rn_n_est[i,:,:], L1, L2, L3, L4, M_pn_n_est[i,:,:])
    
        # 2) Observation Jacobian
        Hn = Hn_Calculator(M_Rn_n_est[i,:,:],L1,L2,L3,L4,M_pn_n_est[i,:,:])
    
        # 3) Kalman Gain
        K = Kalman_Gain_Calculator(M_Pnn[i,:,:],Hn,Rn)
    
        # 4) Tangent plane perturbation
        mu_n_pert = K @ mu_n
    
        # 5) Vector separation
        delta_p,delta_v,delta_R,delta_ab,delta_wb = Split_Correction_Vector(mu_n_pert)
    
        # 6) Mean Update - The sum in R3 is the traditional sum of vectors, but in SO(3) we use the exp map
        M_pn_n_est[i,:,:] = M_pn_n_est[i,:,:] + delta_p
        M_vn_n_est[i,:,:] = M_vn_n_est[i,:,:] + delta_v
        
        # Note here that the perturbation is originally in R3, and we need to transport it to the Manifold
        M_Rn_n_est[i,:,:] = M_Rn_n_est[i,:,:] @ exp_map_from_so3_to_SO3(CartesianSpace2LieAlgebra_Wedge(delta_R))
        
        M_bias_a_n_n[i,:,:] = M_bias_a_n_n[i,:,:] + delta_ab
        M_bias_w_n_n[i,:,:] = M_bias_w_n_n[i,:,:] + delta_wb
    
        # Concatenated sn vector - ILS estimation we need all the variables in R15
        Rn_in_R3 = LieAlgebra2CartesianSpace_Vee(log_map_from_SO3_to_so3(M_Rn_n_est[i,:,:])).reshape(3,1)
    
        sn = np.concatenate([M_pn_n_est[i,:,:], M_vn_n_est[i,:,:], Rn_in_R3, M_bias_a_n_n[i,:,:], M_bias_w_n_n[i,:,:]])
    
        # ===================
        # Covariance Update (Joseph Form)
        # ===================
        # Term (I - K@H)
        ImKH = np.eye(15) - K @ Hn
    
        # Traditional Joseph structure (P_joseph = ImKH @ P @ ImKH.T + K @ R @ K.T)
        # We force the matrix to be positive definite
        P_joseph = ImKH @ M_Pnn[i,:,:] @ ImKH.T + K @ Rn @ K.T
    
        # Application of geometric adjoints of the tangent plane at the boundaries
        adjoint = Adj_SO3(delta_R)
        M_Pnn[i,:,:] = adjoint @ P_joseph @ adjoint.T
        
        # ILS Generation
        LOC, GS = Generator_ILS_Gaussian_Mixture(sn, M_Pnn[i,:,:], sample_size, slope)
        M_ILS_LOC_err[i] = (LOC - Real_ILS_LOC[i])**2
        M_ILS_GS_err[i]  = (GS - Real_ILS_GS[i])**2

    return M_pn_n_est, M_vn_n_est, M_Rn_n_est, M_bias_a_n_n, M_bias_w_n_n, M_Pnn, M_ILS_LOC_err, M_ILS_GS_err

def Plot_LOC_GS_RMSE(N_MC, M_ILS_LOC_err, M_ILS_GS_err):

    # RMSE calculation 
    M_ILS_LOC_err = np.sqrt(1/N_MC * M_ILS_LOC_err)
    M_ILS_GS_err = np.sqrt(1/N_MC * M_ILS_GS_err)
    
    # ====================================================================
    # MATPLOTLIB GLOBAL LATEX FONT CONFIGURATION
    # ====================================================================
    plt.rcParams.update({
        "text.usetex": True,                    # Enables external LaTeX rendering engine
        "font.family": "serif",                 # Uses serif fonts (TeX default)
        "font.serif": ["Computer Modern Roman"],# Classic LaTeX font
        "axes.labelsize": 14,                   # Axis label font size
        "font.size": 12,                        # General font size
        "legend.fontsize": 11,                  # Legend font size
        "xtick.labelsize": 11,                  # X-axis tick font size
        "ytick.labelsize": 11,                  # Y-axis tick font size
        "figure.titlesize": 16                  # Figure title font size
    })
    
    # Time vector based on sampling frequency
    vet_tempo = dt * np.arange(N)
    
    # ====================================================================
    # PLOT 1: LOCALIZER RMSE (ILS LOC) WITH CATEGORY LIMITS (CATS)
    # ====================================================================
    plt.figure(figsize=(7, 4.5))
    if M_ILS_LOC_err.ndim > 1:
        M_ILS_LOC_err_mean = np.nanmean(M_ILS_LOC_err, axis=0)
    else:
        M_ILS_LOC_err_mean = M_ILS_LOC_err
    
    # Filter tracking performance curve
    plt.plot(vet_tempo, M_ILS_LOC_err_mean, color='teal', linewidth=2, label=r"$LOC$ Performance")
    
    # Horizontal baseline references for ILS Categories
    plt.axhline(y=1.0, color='crimson', linestyle=':', linewidth=1.5, label=r"$CAT\ III$ Limit ($1.0$\,m)")
    plt.axhline(y=2.5, color='darkorange', linestyle='--', linewidth=1.5, label=r"$CAT\ II$ Limit ($2.5$\,m)")
    plt.axhline(y=3.5, color='forestgreen', linestyle='-.', linewidth=1.5, label=r"$CAT\ I$ Limit ($3.5$\,m)")
    
    plt.title(r"\textbf{Localizer RMSE} ($LOC$)", color='darkblue')
    plt.xlabel(r"Time [s]")
    plt.ylabel(r"Error [m]")
    plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.legend(loc='best', framealpha=0.9)
    plt.tight_layout()
    plt.savefig('localizer_rmse_catsEKF.pdf', bbox_inches='tight')
    plt.show()
    
    # ====================================================================
    # PLOT 2: GLIDE PATH RMSE (ILS GP) WITH GROUND CUTOFF & EASA BOUNDS
    # ====================================================================
    # Flight time to touchdown estimation
    T = 8000 * np.sqrt(1 + np.tan(np.deg2rad(3))**2) / 70 
    # Safe cutoff limit threshold calculation (stopping 3 seconds prior to impact)
    T_arr = np.floor(T) - 3
    indice_corte = int(T_arr / dt)
    
    plt.figure(figsize=(7, 4.5))
    if M_ILS_GS_err.ndim > 1:
        M_ILS_GS_err_mean = np.nanmean(M_ILS_GS_err, axis=0)
    else:
        M_ILS_GS_err_mean = M_ILS_GS_err
    
    # Truncated tracking plot up to touchdown horizon
    plt.plot(vet_tempo[0:indice_corte], M_ILS_GS_err_mean[0:indice_corte], color='darkorange', linewidth=2, label=r"$GP$ Performance")
    
    # Horizontal baseline reference for EASA regulatory accuracy limits
    plt.axhline(y=0.025, color='crimson', linestyle='--', linewidth=1.5, label=r"EASA Accuracy ($0.025^\circ$)")
    
    plt.title(r"\textbf{Glide Path RMSE} ($GP$)", color='darkblue')
    plt.xlabel(r"Time [s]")
    plt.ylabel(r"Error [$^\circ$]") 
    plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.legend(loc='best', framealpha=0.9)
    plt.tight_layout()
    plt.savefig('glide_path_rmse_easaEKF.pdf', bbox_inches='tight')
    plt.show()

print("================================")
print("Functions Loaded")
print("================================")

Functions Loaded


In [ ]:
                             # ====================================================================
                             #               Lie Group Extended Kalman Filter.                    #
                             #                    Estimation Processing.                          #
                             # ====================================================================



# =====================================================
# Additional data generation
# =====================================================

# Covariance Matrix of the noises
Sa  = sigma_a**2*np.eye(3)
Sw  = sigma_w**2*np.eye(3)
Sba = sigma_ba**2*np.eye(3)
Sbw = sigma_bw**2*np.eye(3)

# Covariance Matrix of the observation noises
C1 = sigma_eps_s1**2 * np.eye(3)
C2 = sigma_eps_s2**2 * np.eye(3)
C3 = sigma_eps_s3**2 * np.eye(3)
C4 = sigma_eps_s4**2 * np.eye(3)
Rn = block_diag(C1,C2,C3,C4)

# Initital covariance Matrix
P0 = np.zeros((15,15))
#P0[0:3,0:3] = (dt**4/4)* R_start @ Sa @ R_start.T
#P0[0:3,3:6] = (dt**3/2)* R_start @ Sa @ R_start.T
#P0[3:6,0:3] = (dt**3/2)* R_start @ Sa @ R_start.T
#P0[3:6,3:6] = (dt**2)* R_start @ Sa @ R_start.T
#P0[6:9,6:9] = (dt**2)*Sw
P0[9:12,9:12] = Sba
P0[12:15,12:15] = Sbw

M_ILS_LOC_Monte_Carlo = np.zeros(N)
M_ILS_GS_Monte_Carlo = np.zeros(N)

# =====================================================
#             MONTE CARLO SIMULATION 
# =====================================================
# Here we will compute the RMSE LOC and GS
# =====================================================

start_time = time.perf_counter()

for exp in range(N_MC):

    print("="*60)
    print("Monte Carlo Experiment",exp)
    print("="*60)
    
    # =====================================================
    # STEP 1 - Synthetic Data Generation
    # =====================================================
    P_true, V_true, M_rotations, bias_a, bias_w, acc_m, gyro_m, y_s1, y_s2, y_s3, y_s4, ILS_LOC, ILS_GS = \
                Synthetic_Data_Generation(dt,T,xi1,xi2,slope,
                                          sigma_a,sigma_w,sigma_ba,sigma_bw,
                                          sigma_eps_s1,sigma_eps_s2,sigma_eps_s3,sigma_eps_s4,
                                          L1,L2,L3,L4,
                                          pos_start,vel_start)

    # =====================================================
    # STEP 2 - Plot real trajectory (optional)
    # =====================================================
    #Plot_3D_trajectory(P_true)

    # =====================================================
    # STEP 3 - Run the estimation algorithm
    # =====================================================
    _, _, _, _, _, _, M_ILS_LOC_err, M_ILS_GS_err = \
    Lie_Group_Extended_Kalman_Filter(dt,T,y_s1,y_s2,y_s3,y_s4,
                                     acc_m,gyro_m,
                                     pos_start,vel_start,
                                     P0,Rn,
                                     L1,L2,L3,L4,
                                     Number_samples,slope,
                                    ILS_LOC,ILS_GS)

    # Store the error - Sum of square errors
    M_ILS_LOC_Monte_Carlo = M_ILS_LOC_Monte_Carlo + M_ILS_LOC_err
    M_ILS_GS_Monte_Carlo = M_ILS_GS_Monte_Carlo + M_ILS_GS_err
    
# =====================================================
# STEP 4 - Plot the Monte Carlo Results
# =====================================================
Plot_LOC_GS_RMSE(N_MC, M_ILS_LOC_Monte_Carlo, M_ILS_GS_Monte_Carlo)

end_time = time.perf_counter()

elapsed_time = end_time - start_time
elapsed_time_hours = elapsed_time / 3600

# Code for printing:
print("============================================")
print(f"End")
print(f"Time: {elapsed_time_hours:.6f} hours")
print("============================================")